# 02 — Market exploration

Context before the spread analysis: how Western Hub prices and PJM load behaved over the exploration window (**Jan 2023 – Jun 2025**, train + validation). The test period (2025 H2) and the 2026 holdout are not used in exploratory analysis, so nothing learned here leaks into the out-of-sample evaluation.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import load_config
from src.analysis.regimes import add_regimes, eda_frame
from src.analysis import spreads as sp, diagnostics as dg, plots as pl

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
pl.apply_style()
cfg = load_config()
FIG = cfg["paths"]["figures"]
full = pd.read_parquet(cfg["paths"]["processed"] / "hourly.parquet")          # holdout file is never loaded
train_mask = full["period"] == "train"
full, thresholds = add_regimes(full, train_mask)       # thresholds fitted on 2023-2024 only
full, ev_thr = dg.extreme_flags(full, train_mask)      # extreme-event thresholds fitted on 2023-2024 only
d = eda_frame(full)                                    # train + validation: Jan 2023 - Jun 2025
d["year"] = d["market_date"].dt.year
d["err_pct"] = d["load_forecast_error"] / d["forecast_load"] * 100
print(f"EDA window: {d.market_date.min().date()} -> {d.market_date.max().date()}, {len(d):,} hours")
print("forecast-load regime cut points (MW, 25/75/95th pct of 2023-24):", thresholds["forecast_load_MW"].round(0))
print("extreme thresholds ($/MWh):", {k: round(float(v), 1) for k, v in ev_thr.items()})

## Price and load levels by year

In [ ]:
d.groupby("year").agg(
    da_mean=("da_lmp", "mean"), rt_mean=("rt_lmp", "mean"), da_std=("da_lmp", "std"), rt_std=("rt_lmp", "std"),
    rt_max=("rt_lmp", "max"), mean_load_GW=("actual_load", lambda s: s.mean() / 1000), peak_load_GW=("actual_load", lambda s: s.max() / 1000))

RT prices are far more volatile than DA prices, even though their averages are almost identical:

In [ ]:
d[["da_lmp", "rt_lmp"]].describe(percentiles=[.01, .05, .5, .95, .99]).T

## Monthly averages

In [ ]:
monthly = d.set_index("timestamp_ept").resample("MS")[["da_lmp", "rt_lmp", "da_rt_spread", "actual_load"]].mean()
monthly.index = monthly.index.strftime("%Y-%m")
monthly.assign(actual_load=monthly.actual_load / 1000).rename(columns={"actual_load": "load_GW"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
x = np.arange(len(monthly))
ax.plot(x, monthly["da_lmp"], color=pl.BLUE, label="Day-ahead")
ax.plot(x, monthly["rt_lmp"], color=pl.ORANGE, label="Real-time")
ax.set_xticks(x[::3], monthly.index[::3])
ax.set_ylabel("Average LMP ($/MWh)")
ax.legend(loc="upper left", ncols=2)
i = int(np.argmax(monthly["rt_lmp"].to_numpy()))
ax.annotate(f"{monthly.index[i]}: RT ${monthly['rt_lmp'].iloc[i]:.0f}", (i, monthly["rt_lmp"].iloc[i]),
            xytext=(-10, 6), textcoords="offset points", ha="right", fontsize=8.5, color=pl.INK_2)
pl._finish(fig, ax, "Monthly average DA and RT prices track each other closely",
           "PJM Western Hub, monthly mean of hourly LMPs.",
           FIG / "00_monthly_prices.png");

## Prices against load

How prices change across load deciles.

In [ ]:
bins = pd.qcut(d["actual_load"], 10)
d.groupby(bins, observed=True).agg(load_GW=("actual_load", lambda s: s.median() / 1000), da_median=("da_lmp", "median"),
    rt_median=("rt_lmp", "median"), rt_p95=("rt_lmp", lambda s: s.quantile(.95)), rt_max=("rt_lmp", "max")).reset_index(drop=True)

In [ ]:
pd.DataFrame({"forecast_load vs DA": dg.correlations(d, "forecast_load", "da_lmp"),
              "actual_load vs RT": dg.correlations(d, "actual_load", "rt_lmp")})

## Summary

* **DA and RT have almost identical average levels, but RT is much more volatile:** std $36.8 vs $24.1, max $1,904 vs $450 (Jan 2023 – Jun 2025).
* **Prices stepped up in 2025:** mean DA $48 vs $33 in 2023–24, and RT std roughly doubled ($63.5 vs $22–28). January 2025 (cold) and June 2025 (heat wave) are the standout months.
* **Prices are convex in load:** median RT rises gently across the first nine load deciles ($18 → $37), then jumps in the top decile (median $52, 95th percentile $186 vs $87 one decile lower).
* **Forecast load and DA price are strongly related** (Spearman 0.67), as are actual load and RT (0.60). Load level sets price level; notebook 03 asks what drives the *gap* between DA and RT.